## 面试问题

循环失败注入测试：工具失败/观察缺失/超预算/乱序返回？

## 回答主线

稳健性不能只测 happy path，要主动注入故障：超时、空返回、超预算、乱序。用可配置故障工具在指定步注入指定故障，验证降级/重试/停止行为。本 Notebook 对比无注入(只跑正常路径)与注入(第1步超时、第2步空返回)，验证循环把超时降级为重试/缓存、把空返回走兜底。

## 真实案例

三步查询循环。注入第1步超时(可恢复)应降级为重试/缓存，第2步空返回(观察缺失)应走兜底。对比只跑正常路径与注入故障。数据为教学故障，不代表真实系统。

In [1]:
QUERIES = ["q0", "q1", "q2"]  # 三个待查询。
print("查询序列:", QUERIES)  # 展示查询。
print("将注入的故障: 第1步超时, 第2步空返回")  # 说明注入计划。

查询序列: ['q0', 'q1', 'q2']
将注入的故障: 第1步超时, 第2步空返回


## 基线（Baseline）

反面基线：只用无故障工具跑正常路径。三步全部成功，但超时、空返回等故障路径从未被测过——线上第一次遇到就可能崩。

In [2]:
def perfect_tool(query, step):  # 无故障工具：总是返回有效结果。
    return {"result": "data_for_" + query, "ok": True}  # 返回正常结果。

def run_loop(tool, queries):  # 运行查询循环。
    outcomes = []  # 记录每步结果。
    for step, q in enumerate(queries):  # 逐步查询。
        r = tool(q, step)  # 调用工具。
        outcomes.append(r.get("result") if r.get("ok") else "degraded")  # 成功记结果否则降级。
    return outcomes  # 返回结果序列。

happy = run_loop(perfect_tool, QUERIES)  # 只跑正常路径。
print("happy path 结果:", happy)  # 展示全部成功但故障路径未测。

happy path 结果: ['data_for_q0', 'data_for_q1', 'data_for_q2']


## 失败案例与修正

只测 happy path 无法暴露故障路径。修正是故障注入：可配置故障工具在第 k 步注入故障，稳健循环对超时降级为重试/缓存、对空返回走兜底，验证降级行为正确。

In [3]:
def make_faulty_tool(faults):  # 构造一个可配置故障的工具。
    def tool(query, step):  # 内部工具函数。
        fault = faults.get(step)  # 查该步注入的故障。
        if fault == "timeout":  # 注入超时故障。
            return {"result": None, "ok": False, "error": "timeout"}  # 返回可恢复的超时。
        if fault == "empty":  # 注入空返回故障。
            return {"result": None, "ok": True, "error": None}  # 返回有效标记但结果为空。
        return {"result": "data_for_" + query, "ok": True}  # 否则正常返回。
    return tool  # 返回配置好的工具。

def run_robust_loop(tool, queries):  # 带降级处理的稳健循环。
    outcomes = []  # 记录每步结果。
    for step, q in enumerate(queries):  # 逐步查询。
        r = tool(q, step)  # 调用工具。
        if not r["ok"] and r.get("error") == "timeout":  # 超时是可恢复错误。
            outcomes.append("retry_or_cache")  # 降级为重试或缓存。
        elif r["result"] is None:  # 观察缺失即空返回。
            outcomes.append("fallback_empty")  # 走空返回兜底。
        else:  # 正常结果。
            outcomes.append(r["result"])  # 记录正常结果。
    return outcomes  # 返回结果序列。

In [4]:
faults = {1: "timeout", 2: "empty"}  # 在第1步注入超时第2步注入空返回。
faulty_tool = make_faulty_tool(faults)  # 构造故障工具。
robust = run_robust_loop(faulty_tool, QUERIES)  # 在注入故障下运行稳健循环。
degraded_count = sum(1 for r in robust if r in ("retry_or_cache", "fallback_empty"))  # 统计降级次数。
print("注入故障后结果:", robust)  # 展示循环对超时和空返回的降级。
print("被降级处理的步数:", degraded_count)  # 展示故障被正确降级而非崩溃。

注入故障后结果: ['data_for_q0', 'retry_or_cache', 'fallback_empty']
被降级处理的步数: 2


In [5]:
print("happy path 全部结果:", happy)  # 正常路径全成功。
print("注入故障后结果:", robust)  # 故障路径被降级处理。
print("被降级步数:", degraded_count, "/ 总步数", len(QUERIES))  # 展示降级比例。
print("两个故障都被降级而非崩溃, 故障路径得到验证")  # 强调稳健性。

happy path 全部结果: ['data_for_q0', 'data_for_q1', 'data_for_q2']
注入故障后结果: ['data_for_q0', 'retry_or_cache', 'fallback_empty']
被降级步数: 2 / 总步数 3
两个故障都被降级而非崩溃, 故障路径得到验证


## 结果解读

happy path 三步全成功但没测到任何故障；注入第1步超时被降级为 `retry_or_cache`、第2步空返回走 `fallback_empty`，两个故障都被正确降级而非崩溃。要点：注入点可配置、区分故障类型走不同分支、验证降级行为正确、故障场景进回归集。

In [6]:
assert happy == ["data_for_q0", "data_for_q1", "data_for_q2"]  # happy path 全部成功。
assert robust[0] == "data_for_q0"  # 第0步正常返回。
assert robust[1] == "retry_or_cache"  # 第1步超时被降级为重试或缓存。
assert robust[2] == "fallback_empty"  # 第2步空返回走兜底。
assert degraded_count == 2  # 注入的两个故障都被降级处理。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
